# Simulating Short Rate and Zero Coupon Bond Yields

In [ ]:
import os
from codelib.file_management.dynamic_file_pathing import get_root
from typing import Union
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from codelib.Johan.visualization.base import fan_chart
from codelib.Models.vasicek_model import VasicekModel

## Setting up

In [ ]:
root = get_root()
data_folder = os.path.join(root, "GBI Optimisation", "Data")
output_folder = os.path.join(root, "Simulation", "Bonds")

np.random.seed(222)
short_rate = os.path.join(root, "Miscellaneous", "Archive", "Data", "short_rate.xlsx")


## Initialise parameters for simulation

In [ ]:
vas = VasicekModel(short_rate,252)
initial_rate=vas.rates.iloc[-1]
kappa, theta, beta = vas.estimate_params()

In [ ]:
rp = -0.2

# correlation
rho = 0.6

# simulation definition
num_sim = 10_000
dt = 1.0 / 252
horizon = 10
num_time_steps = int(horizon / dt)
time_points = np.arange(0, num_time_steps + 1, 1) * dt

## Define functions for simulation

In [ ]:
def simulate_vasicek(initial_short_rate: float, kappa: float, theta: float, beta: float, horizon: float,
                     dt: float = 1.0 / 12, num_sim: int = 10000, z_mat=None):
    """
    simulates short rate processes in a vasicek setting until a given horizon

    Parameters
    ----------

    initial_short_rate:
        initial short rate
    kappa:
        speed of mean reversion.
    theta:
        long term mean of the short rate.
    dt:
        increments in time
    horizon:
        time until maturity/expiry (horizon).
    num_sim:
        number of simulations.
    """
    std_rates = np.sqrt(beta ** 2 / (2 * kappa) * (1 - np.exp(-2 * kappa * dt)))

    num_periods = int(horizon / dt)
    short_rates = np.empty((num_sim, num_periods + 1))
    short_rates[:, 0] = initial_short_rate

    if z_mat is None:
        error_terms = np.random.normal(scale=std_rates, size=(num_sim, num_periods))
    else:
        error_terms = std_rates * z_mat

    for i in range(1, num_periods + 1):
        short_rates[:, i] = theta + (short_rates[:, i - 1] - theta) * np.exp(-kappa * dt) + error_terms[:, i - 1]

    return short_rates


def simulate_risk_drivers(mu: float, sigma: float,
                          initial_rate: float, kappa: float, theta: float, beta: float,
                          rho: float,
                          horizon: float,
                          dt: float = 1.0 / 12,
                          num_sim: int = 10_000):
    """
    Function simulating the risky asset and the short rate.
    """

    # define the number of time steps
    num_time_steps = int(horizon / dt)

    # convert parameters of equity values
    mu_scaled = (mu - 0.5 * sigma ** 2) * dt
    sigma_scaled = sigma * np.sqrt(dt)

    # define innovation correlation matrix
    z_corr_mat = np.array([[1.0, rho], [rho, 1.0]])

    # simulate innovations
    z_mat = np.random.multivariate_normal(np.zeros(2), z_corr_mat, size=(num_sim, num_time_steps))

    # simulate equity prices
    log_ret = mu_scaled + sigma_scaled * z_mat[:, :, 0]

    equity_prices = np.ones((num_sim, num_time_steps + 1))
    equity_prices[:, 1:] = np.exp(np.cumsum(log_ret, axis=1))

    # simulate short rates
    short_rates = simulate_vasicek(initial_short_rate=initial_rate,
                                   kappa=kappa,
                                   theta=theta,
                                   beta=beta,
                                   horizon=horizon,
                                   dt=dt,
                                   num_sim=num_sim,
                                   z_mat=z_mat[:, :, 1])

    return equity_prices, short_rates

In [ ]:
equity_prices, short_rates = simulate_risk_drivers(mu, sigma,
                                                   initial_rate, kappa, theta, beta,
                                                   rho,
                                                   horizon,
                                                   dt,
                                                   num_sim)

## Plot fan chart of short rates

In [ ]:
percentiles = np.percentile(short_rates, [2.5, 5, 10, 25, 50, 75, 90, 95, 97.5], axis=0)

fig, ax = plt.subplots(figsize=(10, 6))
fan_chart(time_points, percentiles, ax=ax, color="navy")
plt.show()

## Export the simulated short rates

In [ ]:
short_rates = pd.DataFrame(short_rates)
#short_rates.to_excel(os.path.join(output_folder, "short_rates_sim.xlsx"))

## Define functions for calculating zero coupon bond prices

In [ ]:
def calculate_zero_coupon_yield(time_to_maturity: Union[float, np.ndarray],
                                initial_short_rate: float,
                                kappa: float,
                                theta: float,
                                beta: float,
                                risk_premium: float):
    y_infty = theta - risk_premium * beta / kappa - beta ** 2 / (2 * kappa ** 2)

    b = 1 / kappa * (1 - np.exp(-kappa * time_to_maturity))
    a = y_infty * (time_to_maturity - b) + beta ** 2 / (4 * kappa) * b ** 2

    return (a + b * initial_short_rate) / time_to_maturity

def calculate_zero_coupon_price(time_to_maturity: Union[float, np.ndarray],
                                initial_short_rate: float,
                                kappa: float,
                                theta: float,
                                beta: float,
                                risk_premium: float):

    y_infty = theta - risk_premium * beta / kappa - beta**2 / (2 * kappa**2)

    b = 1 / kappa * (1 - np.exp(-kappa * time_to_maturity))
    a = y_infty * (time_to_maturity  - b) + beta**2 / (4 * kappa) * b**2

    return np.exp(- a - b * initial_short_rate)

## Calculate zero coupon bond yield for a give time and durations

In [ ]:
days_per_year = 252
years = range(1, 11)        #Years 1 through 10
maturities = range(1, 21)   #Maturities from 1 to 20 years

# Dictionary to store computed yields; keys will be simulation year,
# and each value will be another dictionary with maturities as keys.
zcb_yields = {}

# Loop over each simulation year
for year in years:
    # Compute the index corresponding to the end of the year.
    index = year * days_per_year
    # Ensure the index does not exceed the simulation length
    if index >= short_rates.shape[1]:
        index = short_rates.shape[1] - 1

    # Extract the short rates at the selected time (for all simulation paths)
    sr_at_year = short_rates.iloc[:, index].values

    # Initialise a dictionary for this year's maturities
    zcb_yields[year] = {}

    # Loop over each maturity and compute the zero-coupon yield directly.
    for maturity in maturities:
        # Call the yield function directly
        yield_values = calculate_zero_coupon_yield(maturity, sr_at_year, kappa, theta, beta, rp)
        # Store the computed yields in the dictionary.
        zcb_yields[year][maturity] = yield_values

### Conver yields for a specific year to a DataFrame for inspection

In [ ]:
year_to_inspect = 5
df_year5 = pd.DataFrame(zcb_yields[year_to_inspect])
print(f"Zero-coupon yields for year {year_to_inspect}:\n", df_year5.head())


In [ ]:
# Option 2: Recompute prices from yields via the implied short rate.
zcb_prices_from_yield = {}

for year, yields_dict in zcb_yields.items():
    zcb_prices_from_yield[year] = {}
    for maturity, yield_array in yields_dict.items():
        # Calculate a and b for this maturity
        y_infty = theta - rp * beta / kappa - beta ** 2 / (2 * kappa ** 2)
        b = 1 / kappa * (1 - np.exp(-kappa * maturity))
        a = y_infty * (maturity - b) + beta ** 2 / (4 * kappa) * b ** 2

        # Invert the yield formula to get the implied short rate
        implied_sr = (yield_array * maturity - a) / b

        # Compute the price using the pricing function
        price_array = calculate_zero_coupon_price(maturity, implied_sr, kappa, theta, beta, rp)
        zcb_prices_from_yield[year][maturity] = price_array

# Example: inspect prices for year 5
df_prices_model_year5 = pd.DataFrame(zcb_prices_from_yield[5])
print("Zero-coupon bond prices (via model) for year 5:\n", df_prices_model_year5.head())